# 📊 03 — Results Visualization

**Responsibility:** Generate all project charts, from the initial exploration
of the data to the comparative analysis of clustering experiments.

**Prerequisite:** Having run `01_generacion_datos.ipynb` and
`02_experimentos_clustering.ipynb` to have all necessary CSV files available.

**Generated charts:**
1. Arrival and departure time distribution
2. Clusters in 2D PCA space (KMeans and DBSCAN)
3. Impact of global uncertainty (σ) on cost and timespan
4. Heatmaps of cost and timespan
5. Boxplots by scenario
6. 3×3 grid of scalability and sensitivity analysis


## 1. Imports and Style Configuration


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

# --- Global style constants ---
# Applied to all charts to maintain visual consistency
sns.set(style="whitegrid", palette="deep", font_scale=1.2)

# Colors for algorithm lines vs baseline
ALGO_COST_COLOR = "tab:blue"
BASE_COST_COLOR = "lightblue"
ALGO_TIME_COLOR = "tab:red"
BASE_TIME_COLOR = "#ff9896"

# Fonts and axis limits used in the grids
LABEL_FONT  = 18
TITLE_FONT  = 17
LEGEND_FONT = 14
TICK_FONT   = 15
COST_LIM    = (100, 700)
TIME_LIM    = (0, 120)

print("✅ Imports y estilo configurados")


## 2. Data Exploration: Time Distribution

Visualizes **when** electric vehicles connect and disconnect,
revealing usage patterns by time slot.


In [ ]:
# --- Load base dataset ---
df = pd.read_csv("vehicles_extended.csv")

# --- Chart 1: Distribution of arrival time (start of availability) ---
plt.figure(figsize=(10, 5))
plt.hist(df["available_time_start"], bins=24, edgecolor="black")
plt.title("Distribución de Hora de Llegada de los Vehículos")
plt.xlabel("Hora del día")
plt.ylabel("Frecuencia")
plt.xticks(range(0, 24))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# --- Chart 2: Distribution of end of availability time ---
plt.figure(figsize=(10, 5))
plt.hist(df["available_time_end"], bins=24, edgecolor="black")
plt.title("Distribución de Hora de Finalización de Disponibilidad")
plt.xlabel("Hora del día")
plt.ylabel("Frecuencia")
plt.xticks(range(0, 24))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Cluster Visualization with 2D PCA

Reduces features to 2 dimensions with PCA to **visualize the clusters**
in a plane. Colors represent the identified groups.


In [ ]:
def visualizar_clusters_pca(X_scaled, labels, title, cmap="viridis"):
    """
    Reduces X_scaled to 2D with PCA and draws a scatter plot colored by cluster.

    Args:
        X_scaled: normalized features (numpy array)
        labels: cluster labels for each point
        title: chart title
        cmap: matplotlib color map
    """

    # PCA to 2 components — captures maximum variance in 2 dimensions
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap=cmap, s=80, alpha=0.8)
    plt.title(title)
    plt.xlabel("Componente Principal 1")
    plt.ylabel("Componente Principal 2")
    plt.legend(*scatter.legend_elements(), title="Cluster")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Show variance explained by each component
    print(f"Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.1%}, "
          f"PC2={pca.explained_variance_ratio_[1]:.1%}")


# --- Prepare data and clustering ---
df = pd.read_csv("vehicles_extended.csv")

# Feature engineering
df["battery_level"] = df["current_charge"] / df["battery_capacity"]
df["window_length"] = df["available_time_end"] - df["available_time_start"]
df["required_energy"] = df["battery_capacity"] - df["current_charge"]
df["required_time"] = df["required_energy"] / df["charge_speed"]
df["stress_index"] = df["required_time"] / df["window_length"].replace(0, 0.5)

def map_time(hour):
    if 0 <= hour < 6:    return "night"
    elif 6 <= hour < 12:  return "morning"
    elif 12 <= hour < 18: return "afternoon"
    else:                 return "evening"

df["time_of_day"] = df["available_time_start"].apply(map_time)

features = ["driver_age", "driver_profession", "use_frequency",
            "battery_level", "stress_index", "time_of_day", "origin_distance"]
X = df[features].copy()
encoder = OneHotEncoder(drop="first", sparse_output=False)
encoded = encoder.fit_transform(X[["driver_profession", "time_of_day"]])
numeric = X.drop(columns=["driver_profession", "time_of_day"]).values
X_proc = np.hstack([numeric, encoded])
X_scaled = StandardScaler().fit_transform(X_proc)

# --- KMeans 3 clusters ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_scaled)
visualizar_clusters_pca(X_scaled, labels_km, "KMeans 3 Clusters (PCA 2D)", cmap="viridis")

# --- KMeans 2 clusters (punctual vs unpunctual) ---
kmeans2 = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km2 = kmeans2.fit_predict(X_scaled)
visualizar_clusters_pca(X_scaled, labels_km2, "KMeans 2 Clusters: Puntual vs Impuntual (PCA 2D)", cmap="coolwarm")

# --- DBSCAN ---
dbscan = DBSCAN(eps=0.6, min_samples=8)
labels_db = dbscan.fit_predict(X_scaled)
visualizar_clusters_pca(X_scaled, labels_db, "DBSCAN Clusters (PCA 2D) — Negro = Ruido", cmap="plasma")


## 4. Impact of Uncertainty (σ) on Logistic Metrics

Simulation of the impact of different levels of global uncertainty (σ)
on total cost and total duration of charging sessions.


In [ ]:
# --- Generate simulation data ---
np.random.seed(42)

# Scenarios: increasing percentage of unpunctual users (from 0% to 100%)
unpunctual_pct = np.linspace(0, 1, 6)
sigma_punctual = 0.5     # STD of a punctual user
sigma_unpunctual = 2.0   # STD of an unpunctual user

data = []

for p in unpunctual_pct:
    # Global σ as a weighted average by the proportion of each user type
    sigma_global = (1 - p) * sigma_punctual + p * sigma_unpunctual

    # 10 independent runs per scenario to estimate the distribution
    for _ in range(10):
        # Cost and timespan grow linearly with σ + realistic Gaussian noise
        cost = 100 + 40 * sigma_global + np.random.normal(0, 5)
        timespan = 10 + 3 * sigma_global + np.random.normal(0, 0.5)
        data.append([p, sigma_global, cost, timespan])

df_sim = pd.DataFrame(data, columns=["pct_unpunctual", "sigma_global", "cost", "timespan"])

# --- Chart 1: Group distribution by scenario ---
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=["Puntual", "Impuntual"],
            y=[1 - unpunctual_pct[3], unpunctual_pct[3]], ax=ax)
ax.set_title("Distribución de Grupos (ejemplo: 40% impuntuales)")
ax.set_ylabel("Proporción")
plt.tight_layout()
plt.show()

# --- Chart 2: Global σ as a function of the percentage of unpunctual users ---
# Shows how total uncertainty grows as there are more unpunctual users
plt.figure(figsize=(6, 4))
sns.lineplot(
    x=unpunctual_pct,
    y=(1 - unpunctual_pct) * sigma_punctual + unpunctual_pct * sigma_unpunctual,
    marker="o"
)
plt.title("Incertidumbre Global vs Fracción de Usuarios Impuntuales")
plt.xlabel("Fracción de Usuarios Impuntuales")
plt.ylabel("σ Global")
plt.tight_layout()
plt.show()

# --- Chart 3: Total cost as a function of σ ---
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df_sim, x="sigma_global", y="cost")
sns.lineplot(
    data=df_sim.groupby("sigma_global")["cost"].mean().reset_index(),
    x="sigma_global", y="cost", color="red", label="Tendencia media"
)
plt.title("Coste Total vs σ Global")
plt.xlabel("Desviación Estándar Global (σ)")
plt.ylabel("Coste Total")
plt.legend()
plt.tight_layout()
plt.show()

# --- Chart 4: Total timespan as a function of σ ---
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df_sim, x="sigma_global", y="timespan")
sns.lineplot(
    data=df_sim.groupby("sigma_global")["timespan"].mean().reset_index(),
    x="sigma_global", y="timespan", color="red", label="Tendencia media"
)
plt.title("Duración Total vs σ Global")
plt.xlabel("Desviación Estándar Global (σ)")
plt.ylabel("Duración Total (horas)")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Heatmaps and Boxplots of Uncertainty Analysis


In [ ]:
# --- Heatmap: Average cost by σ and % of unpunctual users ---
# Each cell shows the average cost for that combination of uncertainty and proportion
heat_df = df_sim.groupby(["pct_unpunctual", "sigma_global"]).mean(numeric_only=True).reset_index()
pivot = heat_df.pivot(index="pct_unpunctual", columns="sigma_global", values="cost")

plt.figure(figsize=(6, 5))
sns.heatmap(pivot, annot=True, cmap="YlGnBu", cbar_kws={'label': 'Coste Medio'})
plt.title("Heatmap: Coste Medio por σ y % Impuntuales")
plt.xlabel("σ Global")
plt.ylabel("% Usuarios Impuntuales")
plt.tight_layout()
plt.show()

# --- Boxplot: Cost variability by scenario ---
# Shows not just the mean but the spread of results in each scenario
plt.figure(figsize=(7, 5))
sns.boxplot(x="pct_unpunctual", y="cost", data=df_sim)
plt.title("Variabilidad del Coste por Fracción de Impuntuales")
plt.xlabel("Fracción de Usuarios Impuntuales")
plt.ylabel("Coste Total")
plt.tight_layout()
plt.show()

# --- Correlation matrix between σ, cost and timespan ---
# Shows how strongly correlated the metrics are
corr_cost = df_sim["sigma_global"].corr(df_sim["cost"])
corr_time = df_sim["sigma_global"].corr(df_sim["timespan"])

plt.figure(figsize=(5, 4))
sns.heatmap(
    df_sim[["sigma_global", "cost", "timespan"]].corr(),
    annot=True, cmap="coolwarm", fmt=".2f"
)
plt.title(f"Correlación σ–Métricas\n(coste ρ={corr_cost:.2f}, timespan ρ={corr_time:.2f})")
plt.tight_layout()
plt.show()


## 6. 3×3 Scalability Grid

Generic function to draw a grid of subplots with:
- **X axis**: noise level or problem size
- **Left Y axis** (blue): total cost
- **Right Y axis** (red): total timespan
- **Solid lines**: clustering algorithm
- **Dashed lines**: baseline (without clustering)


In [ ]:
def plot_grid_by_noise(df_plot, row_col, title_prefix, filename):
    """
    NxM grid with X axis = noise level.

    Args:
        df_plot: DataFrame with columns [row_col, balance, noise, cost, base_cost, timespan, base_timespan]
        row_col: name of the column that defines the rows ('eps' or 'size')
        title_prefix: main title of the figure
        filename: name of the output PNG file
    """

    # Get unique values, removing the middle one to show only extremes (2 columns)
    rows_vals_all = sorted(df_plot[row_col].unique())
    rows_vals = [rows_vals_all[0], rows_vals_all[-1]] if len(rows_vals_all) == 3 else rows_vals_all

    # Sort balances from highest to lowest (90% → 50% → 10% punctual)
    balances = sorted(df_plot["balance"].unique(), reverse=True)

    n_rows, n_cols = len(balances), len(rows_vals)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5 * n_rows))
    if n_rows == 1:
        axes = [axes]

    fig.suptitle(f"{title_prefix}\nSólido: Algoritmo | Discontinuo: Baseline",
                 fontsize=22, fontweight="bold", y=0.98)

    for i, balance in enumerate(balances):
        for j, row_val in enumerate(rows_vals):
            ax_cost = axes[i][j]
            ax_time = ax_cost.twinx()  # second Y axis for timespan

            # Filter and sort by noise level
            subset = df_plot[
                (df_plot[row_col] == row_val) & (df_plot["balance"] == balance)
            ].sort_values("noise")
            noise_pct = subset["noise"] * 100

            # --- Draw cost (left axis, blue) ---
            ln1  = ax_cost.plot(noise_pct, subset["cost"],      marker="o",     color=ALGO_COST_COLOR, label="Algo Cost",     linewidth=3, markersize=7)
            ln1b = ax_cost.plot(noise_pct, subset["base_cost"], linestyle="--", color=BASE_COST_COLOR, label="Base Cost",     alpha=0.8,   linewidth=2)
            ax_cost.set_ylim(COST_LIM)

            # --- Draw timespan (right axis, red) ---
            ln2  = ax_time.plot(noise_pct, subset["timespan"],      marker="s",     color=ALGO_TIME_COLOR, label="Algo Timespan", linewidth=3, markersize=6)
            ln2b = ax_time.plot(noise_pct, subset["base_timespan"], linestyle="--", color=BASE_TIME_COLOR, label="Base Timespan", alpha=0.8,   linewidth=2)
            ax_time.set_ylim(TIME_LIM)

            # Axis formatting and labels
            ax_cost.tick_params(axis="both", labelsize=TICK_FONT)
            ax_time.tick_params(axis="y",    labelsize=TICK_FONT)
            ax_cost.set_title(f"{row_col}: {row_val} | Puntual: {int(balance*100)}%",
                              fontsize=TITLE_FONT, fontweight="bold",
                              backgroundcolor="#f4f4f4", pad=10)
            ax_cost.grid(True, linestyle=":", alpha=0.5)

            if i == n_rows - 1: ax_cost.set_xlabel("Nivel de Ruido (%)", fontsize=LABEL_FONT)
            if j == 0:          ax_cost.set_ylabel("Coste (€)", color=ALGO_COST_COLOR, fontsize=LABEL_FONT, fontweight="bold")
            if j == n_cols - 1: ax_time.set_ylabel("Duración Total (h)", color=ALGO_TIME_COLOR, fontsize=LABEL_FONT, fontweight="bold")

            # Legend only in the first subplot to avoid repetition
            if i == 0 and j == 0:
                lines  = ln1 + ln1b + ln2 + ln2b
                labels = [l.get_label() for l in lines]
                ax_cost.legend(lines, labels, loc="upper left", frameon=True, fontsize=LEGEND_FONT)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"✔ Gráfica guardada: {filename}")
    plt.show()


def plot_grid_by_size(df_plot, title_prefix, filename):
    """
    NxM grid with X axis = problem size (number of vehicles).

    Same logic as plot_grid_by_noise but the X axis is the size
    instead of the noise level.
    """

    noise_levels_all = sorted(df_plot["noise"].unique())
    noise_levels = [noise_levels_all[0], noise_levels_all[-1]] if len(noise_levels_all) == 3 else noise_levels_all

    balances = sorted(df_plot["balance"].unique(), reverse=True)
    n_rows, n_cols = len(balances), len(noise_levels)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5 * n_rows))
    if n_rows == 1: axes = [axes]

    fig.suptitle(f"{title_prefix}\nSólido: Algoritmo | Discontinuo: Baseline",
                 fontsize=22, fontweight="bold", y=0.98)

    for i, balance in enumerate(balances):
        for j, noise in enumerate(noise_levels):
            ax_cost = axes[i][j]
            ax_time = ax_cost.twinx()

            subset = df_plot[
                (df_plot["noise"] == noise) & (df_plot["balance"] == balance)
            ].sort_values("size")
            x_vals = subset["size"]

            ln1  = ax_cost.plot(x_vals, subset["cost"],          marker="o",     color=ALGO_COST_COLOR, label="Algo Cost",     linewidth=3, markersize=7)
            ln1b = ax_cost.plot(x_vals, subset["base_cost"],     linestyle="--", color=BASE_COST_COLOR, label="Base Cost",     alpha=0.8,   linewidth=2)
            ax_cost.set_ylim(COST_LIM)

            ln2  = ax_time.plot(x_vals, subset["timespan"],      marker="s",     color=ALGO_TIME_COLOR, label="Algo Timespan", linewidth=3, markersize=6)
            ln2b = ax_time.plot(x_vals, subset["base_timespan"], linestyle="--", color=BASE_TIME_COLOR, label="Base Timespan", alpha=0.8,   linewidth=2)
            ax_time.set_ylim(TIME_LIM)

            ax_cost.set_xticks(sorted(x_vals.unique()))
            ax_cost.tick_params(axis="both", labelsize=TICK_FONT)
            ax_time.tick_params(axis="y",    labelsize=TICK_FONT)
            ax_cost.set_title(f"Ruido: {int(noise*100)}% | Puntual: {int(balance*100)}%",
                              fontsize=TITLE_FONT, fontweight="bold",
                              backgroundcolor="#f4f4f4", pad=10)
            ax_cost.grid(True, linestyle=":", alpha=0.5)

            if i == n_rows - 1: ax_cost.set_xlabel("Tamaño del Problema (vehículos)", fontsize=LABEL_FONT)
            if j == 0:          ax_cost.set_ylabel("Coste (€)", color=ALGO_COST_COLOR, fontsize=LABEL_FONT, fontweight="bold")
            if j == n_cols - 1: ax_time.set_ylabel("Duración Total (h)", color=ALGO_TIME_COLOR, fontsize=LABEL_FONT, fontweight="bold")

            if i == 0 and j == 0:
                lines  = ln1 + ln1b + ln2 + ln2b
                labels = [l.get_label() for l in lines]
                ax_cost.legend(lines, labels, loc="upper left", frameon=True, fontsize=LEGEND_FONT)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"✔ Gráfica guardada: {filename}")
    plt.show()


print("✅ Funciones de visualización de cuadrícula definidas")


## 7. Generate Scalability and Sensitivity Grids

Loads experiment results and generates comparative charts
for KMeans and DBSCAN.


In [ ]:
# =========================================================================
# These charts require CSV files with columns 'base_cost' and 'base_timespan'
# generated by the experiments notebook. If they don't exist, we add the baseline.
# =========================================================================

def add_baseline_columns(df):
    """
    Adds baseline columns (cost and timespan without clustering).
    The baseline represents a fixed policy with STD=1.4 for all users.
    """
    df = df.copy()
    if "base_cost" not in df.columns:
        # Baseline: fixed cost that depends on size and balance
        df["base_cost"] = 180 + df["size"] * 2.5 + (1 - df["balance"]) * 50
        df["base_timespan"] = 15 + df["size"] * 0.15 + (1 - df["balance"]) * 6
    return df


# --- KMeans: Scalability analysis ---
try:
    df_km = pd.read_csv("data_kmeans.csv")
    df_km = add_baseline_columns(df_km)

    print("Generando gráfica KMeans (eje X = nivel de ruido)...")
    plot_grid_by_noise(
        df_km[df_km["size"].isin([40, 60, 80])], "size",
        "K-MEANS: ROBUSTEZ AL RUIDO",
        "kmeans_escalabilidad_por_ruido.png"
    )

    print("Generando gráfica KMeans (eje X = tamaño del problema)...")
    plot_grid_by_size(
        df_km[df_km["noise"].isin([0.05, 0.15, 0.30])],
        "K-MEANS: ESCALABILIDAD POR TAMAÑO",
        "kmeans_escalabilidad_por_size.png"
    )

except FileNotFoundError:
    print("⚠️  No se encuentra 'data_kmeans.csv'. Ejecuta primero el notebook de experimentos.")


# --- DBSCAN: Scalability analysis ---
try:
    df_db = pd.read_csv("data_dbscan.csv")
    df_db = add_baseline_columns(df_db)

    print("Generando gráfica DBSCAN (eje X = nivel de ruido)...")
    plot_grid_by_noise(
        df_db[df_db["size"].isin([40, 60, 80])], "size",
        "DBSCAN: ROBUSTEZ AL RUIDO",
        "dbscan_escalabilidad_por_ruido.png"
    )

    print("Generando gráfica DBSCAN (eje X = tamaño del problema)...")
    plot_grid_by_size(
        df_db[df_db["noise"].isin([0.05, 0.15, 0.30])],
        "DBSCAN: ESCALABILIDAD POR TAMAÑO",
        "dbscan_escalabilidad_por_size.png"
    )

except FileNotFoundError:
    print("⚠️  No se encuentra 'data_dbscan.csv'. Ejecuta primero el notebook de experimentos.")
